In [2]:
from pathlib import Path
from typing import Tuple, List
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from __future__ import annotations
from dataclasses import dataclass

# Load log

In [137]:
class Capability:
    def __init__(self, line, parent):
        #print(line)
        _, boundsOffset, _, boundsLength, _, boundsVirtualBase, _, capPerms   = line.split()
        self.boundsOffset = int(boundsOffset, 16)
        self.boundsLength = int(boundsLength, 16)
        self.boundsVirtBase = int(boundsVirtualBase, 16)
        self.capPerms = int(capPerms, 16)
        self.parent = parent
        parent.child = self

    def capKey(self):
        return (self.boundsVirtBase, self.boundsLength)
    
    def __eq__(self, other):
        return self.boundsOffset == other.boundsOffset and self.boundsLength == other.boundsLength and self.boundsVirtBase == other.boundsVirtBase and self.capPerms == other.capPerms

    def __hash__(self):
        return hash((self.boundsOffset, self.boundsLength, self.boundsVirtBase, self.capPerms))
    
    def __repr__(self):
        return f"Cap virtBase {self.boundsVirtBase} length {self.boundsLength} offset {self.boundsOffset}"

class ReportAccessLog:
    def __init__(self, line, index):
        #44200 Prefetcher logReportAccess level 1 addr 00000000c0007648 pcHash c0000260 hitMiss 1 boundsOffset 0000000000000008 boundsLength 0000000000002f30 boundsVirtBase 00000000c0007640 capPerms 0000000000001111000111111010111

        #print(line.split())
        clock_time,_, _, _, addr, _, pcHash, _, isMiss, _, boundsOffset, _, boundsLength, _, boundsVirtualBase, _, capPerms, _, op   = line.split()
        self.clock_time = int(clock_time)/10
        self.addr = int(addr, 16)
        self.pcHash = int(pcHash, 16)
        self.isMiss = int(isMiss)
        self.op = op
        self.cap = Capability("boundsOffset"+line.partition("op")[0].partition("boundsOffset")[2], self)

        self.index = index
    
class ReportDataArrivalCap:
    def __init__(self, line, parent):
        pre, _, cap = line.partition("boundsOffset")
        _, _, _, _, index, _, tag, _, addr = pre.split()
        self.index = int(index)
        self.tag = bool(int(tag))
        self.addr = int(addr, 16)
        if self.tag:
            self.cap = Capability("boundsOffset"+cap, parent)
        self.parent = parent
        parent.child = self

class ReportDataArrival:
    def __init__(self, line, index):
        clock_time,_, _, _, requestAddr, _, pcHash, _, wasMiss, _, wasPrefetch, _, boundsOffset, _, boundsLength, _, boundsVirtualBase, _, capPerms, _, op  = line.split()
        self.clock_time = int(clock_time)/10
        self.requestAddr = int(requestAddr, 16)
        self.pcHash = int(pcHash, 16)
        self.wasMiss = int(wasMiss)
        self.wasPrefetch = int(wasPrefetch)

        #print(line)
        self.requestCap = Capability("boundsOffset"+line.partition("op")[0].partition("boundsOffset")[2], self)
        self.capabilites: List[ReportDataArrivalCap] = []

        self.sel_capability = None

        self.index = index

    def add_capability(self, line, parent_data_arrivals):
        capability = ReportDataArrivalCap(line, self)
        if capability.tag:
            self.capabilites.append(capability)
            parent_data_arrivals[capability.cap.capKey()] = capability.cap 

        return capability

    def add_sel_capability(self, line):
        capability = ReportDataArrivalCap(line, self)
        if capability.tag:
            self.sel_capability = capability 

        return capability

class NextPrefetchAddr:
    def __init__(self, line):
        clock_time,_, _, addr = line.split()
        self.clock_time = int(clock_time)/10
        self.addr = int(addr, 16)

class TlbResponse:
    def __init__(self, line):
        self.clock_time = int(line.split()[0])/10
        self.paddr = int(line.split()[8].strip(",").strip("'h"), 16)

@dataclass
class PredictionResponseMatch:
    clock_time: str
    predIdxTag: int
    parentOffset: int
    childOffset: int
    confidence: int
    virtBase: int

@dataclass
class BackwardsHit:
    tag: int
    parentVirtBase: int
    parentOffset: int
    childOffset: int
    clock_time: str

@dataclass
class BackwardsAddition:
    parentVirtBase:int
    parentOffset: int
    idxTag: int
    clock_time: str

@dataclass
class TimelinessRdResp:
    valid: bool
    idx: int
    tag: int
    pcHash: int
    clock_time: str
    replacementWay: int

@dataclass
class PredictionReplacement:
    clock_time: str
    idx: int
    tag: int
    newParentOffset: int
    newChildOffset: int
    oldParentOffset: int
    oldChildOffset: int

@dataclass 
class TimelinessReplacement:
    clock_time: str
    repResp: int
    idx: int
    tag: int
    pcHash: int

@dataclass
class PredictionDecrease:
    clock_time: str
    idx: int
    tag: int
    oldConfidence: int
    oldParentOffset: int
    oldChildOffset: int

@dataclass
class PredictionMatchUpdate:
    clock_time: str
    idx: int
    tag: int
    oldConfidence: int

def parse_log(input_file, ignore_first_n_instr, ignore_last_n_instr) -> Tuple[List[ReportAccessLog], List[ReportDataArrival], List[ReportAccessLog]]:
    total_order_events = []
    recent_misses = set()
    parent_access = {} #(cap_virtual_base, cap_size, most recent ReportAccessLog)
    parent_data_arrivals = {} #(cap_virtual_base, cap_size, most recent ReportAccessLog)
    accesses = []
    data_arrivals = []
    access_misses = []

    access_index = 0
    data_arrival_index = 0
    
    instruction_count = 0
    instruction_start_time = 0
    instruction_end_time = 0

    with open(input_file, "r") as fp:
        while True:
            line = fp.readline()
            if not line:
                break

            if "logReportAccess" in line:
                reportAccess = ReportAccessLog(line, access_index)
                total_order_events.append(reportAccess)

                access_index += 1

                if reportAccess.isMiss:
                    recent_misses.add(reportAccess.addr)
                    access_misses.append(reportAccess)
                elif reportAccess.addr in recent_misses:
                    #Skip first hit access after miss
                    recent_misses.remove(reportAccess.addr)
                    continue

                accesses.append(reportAccess)
                parent_access[reportAccess.cap.capKey()] = reportAccess.cap

                if reportAccess.cap.capKey() in parent_data_arrivals:
                    parent = parent_data_arrivals[reportAccess.cap.capKey()]
                    reportAccess.parent = parent
                    parent.child = reportAccess
                    #parent.child = reportAccess
                    #print("found parent")

                # if reportAccess.cap.boundsLength == 40000:
                #         print("40000 arrival")
                #         print(reportAccess.cap.capKey())
                #print(reportAccess.cap.boundsLength)
            
            elif "logReportDataArrival" in line:
                data_arrival = ReportDataArrival(line, data_arrival_index)
                total_order_events.append(data_arrival)
                data_arrival_index += 1 

                data_arrival.add_capability(fp.readline(), parent_data_arrivals)
                data_arrival.add_capability(fp.readline(), parent_data_arrivals)
                data_arrival.add_capability(fp.readline(), parent_data_arrivals)
                data_arrival.add_capability(fp.readline(), parent_data_arrivals)

                data_arrival.add_sel_capability(fp.readline())

                data_arrivals.append(data_arrival)

                if data_arrival.requestCap.capKey() in parent_access:
                    # if data_arrival.requestCap.boundsLength == 40000:
                    #     print("40000 arrival in arrival")
                    parent = parent_access[data_arrival.requestCap.capKey()]
                    data_arrival.parent = parent
                    parent.child = data_arrival
                    #parent.child = data_arrival
                    #print("found parent2")

                # if data_arrival.requestCap.boundsLength == 40000:
                #     print(data_arrival.requestCap.capKey())
                    
            elif "getNextPrefetchAddr" in line:
                getNextPrefetchAddr = NextPrefetchAddr(line)
                total_order_events.append(getNextPrefetchAddr)

            elif "got TLB response" in line:
                tlbResponse = TlbResponse(line)
                total_order_events.append(tlbResponse)

            elif "processPredictionResponse tag match and valid offset" in line:
                clock_time, _, _, _, _, _, _, _, _, predIdxTag, _, parentOffset, _, childOffset, _, confidence, _, virtBase = line.split()
                predIdxTag = int(predIdxTag, 16)
                parentOffset = int(parentOffset, 16)
                childOffset = int(childOffset, 16)
                confidence = int(confidence, 16)
                virtBase = int(virtBase, 16)

                predResponse = PredictionResponseMatch(clock_time, predIdxTag, parentOffset, childOffset, confidence, virtBase)
                total_order_events.append(predResponse)

            elif "backwards table hit" in line:
                clock_time, _, _, _, _, _, tag, _, parentVirtBase, _, parentOffset, _, childOffset = line.split()
                tag = int(tag, 16)
                parentVirtBase = int(parentVirtBase, 16)
                parentOffset = int(parentOffset, 16)
                childOffset = int(childOffset, 16)
                backwardsHit = BackwardsHit(tag, parentVirtBase, parentOffset, childOffset, clock_time)
                total_order_events.append(backwardsHit)

            elif "added to backwards table" in line:
                clock_time, _, _, _, _, _, _, _, parentVirtBase, _, parentOffset, _, idx, _, childTag = line.split()
                parentVirtBase = int(parentVirtBase, 16)
                parentOffset = int(parentOffset, 16)

                idxTag = int(childTag + idx, 16)
                backwardsAddition = BackwardsAddition(parentVirtBase, parentOffset, idxTag, clock_time)

                total_order_events.append(backwardsAddition) 

            elif "timeliness table rdResp" in line:
                clock_time, _, _, _, _, _, valid, _, idx, _, tag, _, pcHash, _, repWay, _, _ = line.partition("fshow")[0].split()
                valid = int(valid)
                idx = int(idx, 16)
                tag = int(tag, 16)
                pcHash = int(pcHash, 16)
                timelinessRdResp = TimelinessRdResp(valid, idx, tag, pcHash, clock_time, repWay)
                total_order_events.append(timelinessRdResp)
            
            elif "processPredictionReplacementRd replacement" in line:
                clock_time, _, _, _, _, idx, _, tag, _, newParentOffset, _, newChildOffset, _, oldParentOffset, _, oldChildOffset = line.split()
                idx = int(idx, 16)
                tag = int(tag, 16)
                newParentOffset = int(newParentOffset, 16)
                newChildOffset = int(newChildOffset, 16)
                oldParentOffset = int(oldParentOffset, 16)
                oldChildOffset = int(oldChildOffset, 16)
                predictionReplacement = PredictionReplacement(clock_time, idx, tag, newParentOffset, newChildOffset, oldParentOffset, oldChildOffset)
                total_order_events.append(predictionReplacement)

            elif "processPredictionReplacementRd decrease" in line:
                clock_time, _, _, _, _, _, oldConfidence, _, idx, _, tag, _, oldParentOffset, _, oldChildOffset = line.split()
                idx = int(idx, 16)
                tag = int(tag, 16)
                oldConfidence = int(oldConfidence, 16)
                oldParentOffset = int(oldParentOffset, 16)
                oldChildOffset = int(oldChildOffset, 16)

                predictionDecrease = PredictionDecrease(clock_time, idx, tag, oldConfidence, oldParentOffset, oldChildOffset)
                total_order_events.append(predictionDecrease)

            elif "table miss" in line:
                for i in range(len(total_order_events)-1, 0, -1):
                    if isinstance(total_order_events[i], TimelinessRdResp):
                        total_order_events.pop(i)
                        break

            elif "timeliness replacement" in line:
                clock_time, _, _, _, _, repRepl, _, _, _, idx, _, tag, _, pcHash = line.split()
                repRepl = int(repRepl)
                idx = int(idx, 16)
                tag = int(tag, 16)
                pcHash = int(pcHash, 16)
                timelinessRepl = TimelinessReplacement(clock_time, repRepl, idx, tag, pcHash)
                total_order_events.append(timelinessRepl)

            elif "processPredictionReplacementRd match" in line:
                clock_time, _, _, _, _, _, oldConfidence, _, idx, _, tag = line.split()
                oldConfidence = int(oldConfidence, 16)
                idx = int(idx, 16)
                tag = int(tag, 16)
                predictionMatchUpdate = PredictionMatchUpdate(clock_time, idx, tag, oldConfidence)
                total_order_events.append(predictionMatchUpdate)    
        
            if "RVFI Order" in line:
                instruction_count += 1
                clock_time = int(line.split()[0].strip(":"))/10
                if instruction_count == ignore_first_n_instr:
                    instruction_start_time = clock_time
                if instruction_count == ignore_last_n_instr:
                    instruction_end_time = clock_time

    print(f"access {len(accesses)} misses {len(access_misses)}")
    print(f"IPC {(ignore_last_n_instr - ignore_first_n_instr)/ (instruction_end_time - instruction_start_time)}")
    print(f"Instruction count {instruction_count} num of cycles {instruction_end_time - instruction_start_time}")
    return accesses, data_arrivals, access_misses, total_order_events

In [ ]:
"              798620 prefetecher processPredictionReplacementRd match +1 oldConfidence 00000001 idx 76 tag c00027"

In [ ]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/Uni/Dissertation/DE10Pro-cheri-bgas/bluespec/sim-utils/simulations/prefetch_patricia/sim_0.0/sim_stdout_without_prefetcher")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

In [16]:
misses = {}
miss_times = []
for event in total_order_events[:300000]:
    if isinstance(event, ReportAccessLog):
        #prefetcher.reportRequest(event)
        if event.isMiss:
            misses[event.addr] = event.clock_time
    elif isinstance(event, ReportDataArrival):
          if event.wasMiss and not event.wasPrefetch:
            miss_times.append(event.clock_time - misses[event.requestAddr])
    # #     if event.wasPrefetch:
    # #         prefetches[event.requestAddr] = event.clock_time
    # elif isinstance(event, NextPrefetchAddr):
    #     prefetches[event.addr] = event.clock_time
    # else:
    #     raise Exception()
# timeliness = np.array(timeliness)
# timelines_opposite = np.array(timelines_opposite)

In [ ]:
np.mean(miss_times)

In [ ]:
plt.hist(miss_times, density=False, bins=30)

In [ ]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/Uni/Dissertation/DE10Pro-cheri-bgas/bluespec/sim-utils/simulations/prefetch_patricia/sim_0.0/sim_stdout_fixedTLB_confidence2_size32")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

In [ ]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/Uni/Dissertation/DE10Pro-cheri-bgas/bluespec/sim-utils/simulations/prefetch_patricia/sim_0.0/sim_stdout_fixedTLB_confidence3")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

In [ ]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/Uni/Dissertation/DE10Pro-cheri-bgas/bluespec/sim-utils/simulations/prefetch_patricia/sim_0.0/sim_stdout")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

In [ ]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/Uni/Dissertation/DE10Pro-cheri-bgas/bluespec/sim-utils/simulationv2/small_prediction/prefetch_patricia/sim_0.0/sim_stdout")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

In [ ]:
# #input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
# input_file = Path("sim_stdout_half_copy")

# accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,590000)

In [8]:
tlb_responses = set()
for event in total_order_events[:300000]:
    if isinstance(event, TlbResponse):
        tlb_responses.add(hex(event.paddr))

In [ ]:
tlb_responses

In [13]:
tlb_weirdness = defaultdict(set)
for data_arrival in data_arrivals:
    if data_arrival.requestAddr == data_arrival.requestCap.boundsVirtBase:
        tlb_weirdness[hex(data_arrival.requestAddr)].add(data_arrival.requestCap.boundsOffset)
        #print(hex(data_arrival.requestAddr))
    # print(data_arrival.requestAddr)
    # print(data_arrival.requestCap.boundsVirtBase)

In [ ]:
tlb_weirdness.values()

In [25]:
prefetches = {}
timeliness = []
timelines_opposite = []
for event in total_order_events[:300000]:
    if isinstance(event, ReportAccessLog):
        #prefetcher.reportRequest(event)
        if event.addr in prefetches:
            timeliness.append(event.clock_time - prefetches[event.addr])
    elif isinstance(event, ReportDataArrival):
        pass
    #     if event.wasPrefetch:
    #         prefetches[event.requestAddr] = event.clock_time
    elif isinstance(event, NextPrefetchAddr):
        prefetches[event.addr] = event.clock_time
    # else:
    #     raise Exception()
timeliness = np.array(timeliness)
timelines_opposite = np.array(timelines_opposite)

In [ ]:
data_arrivals[100].pcHash

In [ ]:
plt.hist(timeliness[timeliness<200], density=False, bins=30)

In [ ]:
np.min(timeliness)

In [ ]:
np.median(timeliness)

In [ ]:
aligments = set()
for data_arrival in data_arrivals[1000:]:
    aligments.add(data_arrival.requestCap.boundsOffset % 16)
aligments

In [ ]:
getIndexBits(100, 0, 100)

# Software prefetcher

In [204]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/sim_stdout_no_prefetcher_good")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

access 409112 misses 409112
IPC 3.641561137259543
Instruction count 1119854 num of cycles 54921.5


In [138]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/DisserationAnalysisOffline/sim_stdout_fully_fixed_timeliness_v2")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

access 140152 misses 11255
IPC 3.452788299191012
Instruction count 411347 num of cycles 57924.2


In [97]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/DisserationAnalysisOffline/sim_stdout_fully_fixed_timeliness_v2")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

access 140152 misses 11255
IPC 3.452788299191012
Instruction count 411347 num of cycles 57924.2


In [65]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/DisserationAnalysisOffline/sim_stdout_fully_fixed_timeliness")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

access 123051 misses 9567
IPC 3.452788299191012
Instruction count 362702 num of cycles 57924.2


In [ ]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/Uni/Dissertation/DE10Pro-cheri-bgas/bluespec/sim-utils/simulations/prefetch_patricia/sim_0.0/sim_stdout")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

In [34]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/Uni/Dissertation/DE10Pro-cheri-bgas/bluespec/sim-utils/simulations/prefetch_patricia/sim_0.0/sim_stdout")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

access 224108 misses 5470
IPC 5.620851109274966
Instruction count 682011 num of cycles 35581.8


In [80]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/Uni/Dissertation/DE10Pro-cheri-bgas/bluespec/sim-utils/simulations/prefetch_patricia/sim_0.0/sim_stdout")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

access 224108 misses 5470
IPC 5.620851109274966
Instruction count 682011 num of cycles 35581.8


In [58]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/Uni/Dissertation/DE10Pro-cheri-bgas/bluespec/sim-utils/simulations/prefetch_patricia/sim_0.0/sim_stdout")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

Found!!!
access 224108 misses 5470
IPC 5.620851109274966
Instruction count 682011 num of cycles 35581.8


In [57]:
for event in total_order_events[8200:10000]:
    if isinstance(event, ReportAccessLog):
        print(event.clock_time)

In [ ]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/Uni/Dissertation/DE10Pro-cheri-bgas/bluespec/sim-utils/simulations/prefetch_patricia/sim_0.0/sim_stdout")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

In [ ]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/Uni/Dissertation/DE10Pro-cheri-bgas/bluespec/sim-utils/simulations/prefetch_patricia/sim_0.0/sim_stdout")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

In [ ]:
#input_file = Path("../simulations/adpcm_decode") / "sim_0.0" / "sim_stdout"
input_file = Path("/home/tim/Documents/Uni/Dissertation/DE10Pro-cheri-bgas/bluespec/sim-utils/simulations/prefetch_patricia/sim_0.0/sim_stdout")

accesses, data_arrivals, access_misses, total_order_events = parse_log(input_file, 100000,300000)

In [206]:
from collections import deque
from functools import partial


def getIndexBits(tag, bitsLowerBound, bitsUpperBound):
    return ((tag & (2**(bitsUpperBound) - 1)) >> bitsLowerBound) 

class BackwardTableEntry:
    def __init__(self, parentVirtBase, parentOffset, parentTime, tag):
        self.parentVirtBase = parentVirtBase
        self.parentOffset = parentOffset
        self.parentTime = parentTime
        self.tag = tag

class TimelinessTableEntry:
    def __init__(self, pcHash, clock_time, tag):
        self.pcHash = pcHash
        self.clockTime = clock_time
        self.tag = tag
    def __str__(self):
        return f"tag: {self.tag}, pcHash: {self.pcHash}"
    def __repr__(self):
        return f"tag: {self.tag}, pcHash: {self.pcHash}"

        
class PredictionTableEntry:
    def __init__(self, parentOffset, childOffset, confidence, tag):
        self.parentOffset = parentOffset
        self.childOffset = childOffset
        self.confidence = confidence
        self.tag = tag

class PrefetchRequest:
    def __init__(self, addr, time, childOffset, pcHash):
        self.addr = addr
        self.time = time
        self.childOffset = childOffset
        self.pcHash = pcHash

class ConfidenceUpdateTableEntry:
    def __init__(self, pcHash, parentOffset, childAddr):
        self.pcHash = pcHash
        self.parentOffset = parentOffset
        self.childAddr = childAddr

class BaseBackwardsPrefetcher:
    def __init__(self, timelinessWays, predictionWays, predictionReplacementConfidence, predictionPrefetchConfidence, backwardsTableBits, backwardsTableLowerBits, predictionInReportAccess, timelinessTableBits, predictionTableBits):
        self.backwardsTable = {}
        self.timelinessTable = defaultdict(partial(deque, maxlen=timelinessWays))
        self.timelinessReplacement = defaultdict(int)
        self.predictionTable = defaultdict(list)

        self.predictionWays = predictionWays
        self.predictionReplacmentConfidence = predictionReplacementConfidence
        self.predictionPrefetchConfidence = predictionPrefetchConfidence

        self.totalCacheMisses = 0
        self.prefetchesNotMisses = 0
        self.prefetchesMisses = 0
        
        self.backwardsCollisions = 0
        
        self.timelinessCountParent = []
        self.timelinessCountChild = []

        self.prefetches = []

        self.backwardsTableBits = backwardsTableBits
        self.backwardsTableLowerBits = backwardsTableLowerBits

        self.confidenceUpdateTable = {}

        self.confidenceUpdateCollisions = 0

        self.predictionInReportAccess = predictionInReportAccess
        self.timelinessTableBits = timelinessTableBits
        self.predictionTableBits = predictionTableBits

        self.timelinessWays = timelinessWays

        self.delay_queue = []

    def getBackwardsTableIndex(self, virtBase):
        # print(f"tag {tag} index {getIndexBits(tag, self.backwardsTableLowerBits, self.backwardsTableBits)}")
        return getIndexBits(virtBase, 0, self.backwardsTableBits)

    def getBackwardsTableTag(self, virtBase):
        return getIndexBits(virtBase, self.backwardsTableBits, 64)

    def getTimelinessTableIndex(self, virtBase):
        return getIndexBits(virtBase, 0, self.timelinessTableBits)

    def getTimelinessTableTag(self, virtBase):
        return getIndexBits(virtBase, self.timelinessTableBits, 64)

    def getPredictionTableIndex(self, pcHash):
        return getIndexBits(pcHash, 0, self.predictionTableBits)

    def getPredictionTableTag(self, pcHash):
        return getIndexBits(pcHash, self.predictionTableBits, 32)

    def getConfidenceUpdateTableBits(self, tag):
        return tag

    def nextReplacementWay(self, current):
        if current == self.timelinessWays -1:
            return 0
        else:
            return current + 1

    def addToPredictionTable(self, pcHash, parentOffset, childOffset, clock_time):
        ways = self.predictionTable[self.getPredictionTableIndex(pcHash)]
        
        # if len(ways)==0:
        #     self.predictionTable[pcHash] = [PredictionTableEntry(parentOffset, childOffset, 1)]
        #     return
        
        lowestConfidence = 100000000000000
        lowestConfidenceIndex = -1

        if len(ways) < self.predictionWays:
            #ways.append(PredictionTableEntry(parentOffset, childOffset, 1, self.getPredictionTableTag(pcHash)))
            for i in range(self.predictionWays - len(ways)):
                ways.append(PredictionTableEntry(0, 0, 0, 0))
            self.predictionTable[self.getPredictionTableIndex(pcHash)] = ways
            #return

        way: PredictionTableEntry
        for i, way in enumerate(ways):
            if way.parentOffset == parentOffset and way.childOffset == childOffset and way.tag == self.getPredictionTableTag(pcHash):
                softwarePredictionMatchUpdate.append(PredictionMatchUpdate(clock_time, self.getPredictionTableIndex(pcHash), self.getPredictionTableTag(pcHash), way.confidence))
                print(f"software prediction match +1 {softwarePredictionMatchUpdate[-1]}")
                way.confidence += 1
                ways[i] = way
                self.predictionTable[self.getPredictionTableIndex(pcHash)] = ways
                return

            if way.confidence < lowestConfidence:
                lowestConfidence = way.confidence
                lowestConfidenceIndex = i
        
       


        # Replace if below confidence 
        if lowestConfidence < self.predictionReplacmentConfidence:
            # print(f"software prediction replacment {PredictionReplacement(clock_time, self.getPredictionTableIndex(pcHash), self.getPredictionTableTag(pcHash), parentOffset, childOffset, ways[lowestConfidenceIndex].parentOffset,ways[lowestConfidenceIndex].childOffset )}")
            softwarePredictionReplacement.append(PredictionReplacement(clock_time, self.getPredictionTableIndex(pcHash), self.getPredictionTableTag(pcHash), parentOffset, childOffset, ways[lowestConfidenceIndex].parentOffset,ways[lowestConfidenceIndex].childOffset ))
            ways[lowestConfidenceIndex] = PredictionTableEntry(parentOffset, childOffset, 1, self.getPredictionTableTag(pcHash))
        else:
            # Otherwise decrease confidence for all other ways (is all the correct approach?)
            for i, way in enumerate(ways):
                softwarePredictionDecrease.append(PredictionDecrease(clock_time, self.getPredictionTableIndex(pcHash), self.getPredictionTableTag(pcHash), way.confidence, way.parentOffset, way.childOffset))
                # print(f"software prediction decrease {softwarePredictionDecrease[-1]}")
                
                way.confidence -= 1
                ways[i] = way
        
        self.predictionTable[self.getPredictionTableIndex(pcHash)] = ways

    def reportRequest(self, report_access: ReportAccessLog):
        remaning_delay_queue = []
        for el in self.delay_queue:
            if el[0] <= report_access.clock_time:
                el[1](*el[2])
            else:
                remaning_delay_queue.append(el)
        self.delay_queue = remaning_delay_queue
        
        if self.predictionInReportAccess:
            if report_access.isMiss:
                self.totalCacheMisses += 1
                if self.getBackwardsTableIndex(report_access.cap.boundsVirtBase) in self.backwardsTable:              
                    if self.backwardsTable[self.getBackwardsTableIndex(report_access.cap.boundsVirtBase)].tag == self.getBackwardsTableTag(report_access.cap.boundsVirtBase):

                        parentVirtBase = self.backwardsTable[self.getBackwardsTableIndex(report_access.cap.boundsVirtBase)].parentVirtBase
                        parentOffset = self.backwardsTable[self.getBackwardsTableIndex(report_access.cap.boundsVirtBase)].parentOffset
                        
                        #print(f"software backwards hit {BackwardsHit(self.getBackwardsTableTag(report_access.cap.boundsVirtBase), parentVirtBase, parentOffset, report_access.cap.boundsOffset, report_access.clock_time) }")
                        softwareBackwardsHit.append(BackwardsHit(report_access.cap.boundsVirtBase, parentVirtBase, parentOffset, report_access.cap.boundsOffset, report_access.clock_time))

                        timeliness_queue = self.timelinessTable[self.getTimelinessTableIndex(parentVirtBase)]

                        # print([hex(x.pcHash) for x in timeliness_queue])
                        if len(timeliness_queue) > 0:
                        # print(f"timeliness queue length {len(timeliness_queue)}")
                            # print(timeliness_queue)
                            for i in range(len(timeliness_queue)):
                                if timeliness_queue[i].tag == self.getTimelinessTableTag(parentVirtBase):
                                    pcToPrefetchOn = timeliness_queue[i].pcHash # getting oldest value for simple prefetcher
                                    softwareTimelinessResponse.append(TimelinessRdResp(1, self.getTimelinessTableIndex(parentVirtBase), self.getTimelinessTableTag(parentVirtBase), pcToPrefetchOn, report_access.clock_time, str(self.timelinessReplacement[self.getTimelinessTableIndex(parentVirtBase)])))
                                    # print(f"software TimelinessRdResp {TimelinessRdResp(1, self.getTimelinessTableTag(parentVirtBase), pcToPrefetchOn, report_access.clock_time, self.timelinessReplacement[self.getTimelinessTableIndex(parentVirtBase)])}")

                                    self.delay_queue.append((report_access.clock_time + 3, self.addToPredictionTable, (pcToPrefetchOn, parentOffset, report_access.cap.boundsOffset, report_access.clock_time)))
                                    break
                    else:
                        self.backwardsCollisions += 1

        self.timelinessTable[self.getTimelinessTableIndex(report_access.cap.boundsVirtBase)].append(TimelinessTableEntry(report_access.pcHash, report_access.clock_time, self.getTimelinessTableTag(report_access.cap.boundsVirtBase)))
        softwareTimelinessReplacement.append(TimelinessReplacement(report_access.clock_time, self.timelinessReplacement[self.getTimelinessTableIndex(report_access.cap.boundsVirtBase)], self.getTimelinessTableIndex(report_access.cap.boundsVirtBase), self.getTimelinessTableTag(report_access.cap.boundsVirtBase), report_access.pcHash))
        # print(f"software {softwareTimelinessReplacement[-1]}")
        self.timelinessReplacement[self.getTimelinessTableIndex(report_access.cap.boundsVirtBase)] = self.nextReplacementWay(self.timelinessReplacement[self.getTimelinessTableIndex(report_access.cap.boundsVirtBase)])
       

        way: PredictionTableEntry
        for way in self.predictionTable[self.getPredictionTableIndex(report_access.pcHash)]:
            if way.confidence >= self.predictionPrefetchConfidence and way.tag == self.getPredictionTableTag(report_access.pcHash):
                if way.parentOffset < report_access.cap.boundsLength:
                    softwarePredictionMatches.append(PredictionResponseMatch(report_access.clock_time, report_access.pcHash, way.parentOffset, way.childOffset, way.confidence, report_access.cap.boundsVirtBase))
                    # print(f"software prediction match {softwarePredictionMatches[-1]}")
                    self.prefetches.append(PrefetchRequest(report_access.cap.boundsVirtBase + way.parentOffset, report_access.clock_time, way.childOffset, report_access.pcHash))
            
        
    def reportDataArrival(self, data_arrival: ReportDataArrival):
        # Build backwards table
        # TODO: should I check whether it is sub capability?

        # if not self.predictionInReportAccess:
        #     childVirtBase = data_arrival.requestCap.boundsVirtBase

        #     if data_arrival.wasMiss:
        #         self.totalCacheMisses += 1
        #         if self.getBackwardsTableIndex(childVirtBase) in self.backwardsTable:
                
                
        #             if self.backwardsTable[self.getBackwardsTableIndex(data_arrival.requestCap.boundsVirtBase)].childVirtBase == childVirtBase:
        #                 parentVirtBase = self.backwardsTable[self.getBackwardsTableIndex(childVirtBase)].parentVirtBase
        #                 parentOffset = self.backwardsTable[self.getBackwardsTableIndex(childVirtBase)].parentOffset
                        
        #                 timeliness_queue = self.timelinessTable[self.getTimelinessTableIndex(parentVirtBase)]
        #                 if len(timeliness_queue) > 0:
        #                     for i in range(len(timeliness_queue)):
        #                         if timeliness_queue[i].tag == self.getTimelinessTableTag(parentVirtBase):
        #                             pcToPrefetchOn = timeliness_queue[i].pcHash # getting oldest value for simple prefetcher
        #                             self.addToPredictionTable(pcToPrefetchOn, parentOffset, data_arrival.requestCap.boundsOffset)
        #                             break
        #             else:
        #                 self.backwardsCollisions += 1

        if data_arrival.requestAddr & 0b1111 != 0 and data_arrival.sel_capability:
            print(f"WEIRDNESS addr[3:0] != 0, clock_time {data_arrival.clock_time} request_addr {data_arrival.requestAddr}")

        if data_arrival.requestAddr & 0b1111 == 0 and not data_arrival.wasPrefetch:
            if data_arrival.sel_capability:
                parentVirtBase = data_arrival.requestCap.boundsVirtBase
                parentLength = data_arrival.requestCap.boundsLength
                childVirtBase = data_arrival.sel_capability.cap.boundsVirtBase
                childLength = data_arrival.sel_capability.cap.boundsVirtBase

                if (childVirtBase != parentVirtBase):
                    # print(f"software backwards addition {BackwardsAddition(data_arrival.requestCap.boundsVirtBase, 
                                                                                                            # data_arrival.requestCap.boundsOffset, data_arrival.sel_capability.cap.boundsVirtBase, data_arrival.clock_time)}")
                    softwareBackwardsAddition.append(BackwardsAddition(data_arrival.requestCap.boundsVirtBase, 
                                                                                                            data_arrival.requestCap.boundsOffset, data_arrival.sel_capability.cap.boundsVirtBase, data_arrival.clock_time))
                    self.backwardsTable[self.getBackwardsTableIndex(data_arrival.sel_capability.cap.boundsVirtBase)] = BackwardTableEntry(data_arrival.requestCap.boundsVirtBase, 
                                                                                                            data_arrival.requestCap.boundsOffset, 
                                                                                                            data_arrival.clock_time, self.getBackwardsTableTag(data_arrival.sel_capability.cap.boundsVirtBase))
        prefetchHit = None
        for prefetch in self.prefetches:
            #print(f"{prefetch.addr} {data_arrival.requestAddr}")
            if prefetch.addr == data_arrival.requestAddr:
                prefetchHit = prefetch
                if prefetch.childOffset and data_arrival.sel_capability:
                    childAddr = data_arrival.sel_capability.cap.boundsVirtBase + prefetch.childOffset
                    self.prefetches.append(PrefetchRequest(childAddr, data_arrival.clock_time, None, prefetch.pcHash))
                    self.confidenceUpdateTable[self.getConfidenceUpdateTableBits(childAddr)] = ConfidenceUpdateTableEntry(prefetch.pcHash, data_arrival.requestCap.boundsOffset, childAddr)
                self.prefetches.remove(prefetch)
        
        # if prefetchHit == True:        
        #     print(f"Prefetch match wasMiss {data_arrival.wasMiss}")
        
        if prefetchHit and data_arrival.wasMiss:
            self.prefetchesMisses += 1
            if prefetchHit.childOffset == None:
                self.timelinessCountParent.append(data_arrival.parent.parent.clock_time - (data_arrival.clock_time - data_arrival.parent.parent.clock_time + prefetchHit.time))

                # Update condifidence
                # if data_arrival.requestAddr in self.confidenceUpdateTable:
                #     confidenceUpdateEntry: ConfidenceUpdateTableEntry
                #     confidenceUpdateEntry = self.confidenceUpdateTable[self.getConfidenceUpdateTableBits(data_arrival.requestAddr)]
                #     if confidenceUpdateEntry.childAddr == data_arrival.requestAddr:
                #         ways = self.predictionTable[self.getPredictionTableIndex(confidenceUpdateEntry.pcHash)]
                #         for i, way in enumerate(ways):
                #             way: PredictionTableEntry
                #             if way.parentOffset == confidenceUpdateEntry.parentOffset and way.childOffset == data_arrival.requestCap.boundsOffset:
                #                 way.confidence += 1
                #                 ways[i] = way
                #             else:
                #                 way.confidence -= 1
                #                 ways[i] = way
                #         self.predictionTable[confidenceUpdateEntry.pcHash] = ways
                #     else:
                #         self.confidenceUpdateCollisions += 1
                    

            else:
                self.timelinessCountChild.append(data_arrival.parent.parent.clock_time - (data_arrival.clock_time - data_arrival.parent.parent.clock_time + prefetchHit.time))


        if prefetchHit and not data_arrival.wasMiss:
            self.prefetchesNotMisses += 1

        # On miss 
    def reportStatistics(self):
        coverage = self.prefetchesMisses / self.totalCacheMisses
        accuracy = self.prefetchesMisses / (self.prefetchesMisses + len(self.prefetches))

        return coverage, accuracy, self.totalCacheMisses, self.prefetchesNotMisses, self.prefetchesMisses, len(self.prefetches), np.median(self.timelinessCountParent), np.median(self.timelinessCountChild), np.median(self.timelinessCountParent + self.timelinessCountChild)

In [209]:
def run_prefetcher(prefetcher):
    counter = 0
    for event in total_order_events[:300000]:
        if isinstance(event, ReportAccessLog):
            prefetcher.reportRequest(event)
        elif isinstance(event, ReportDataArrival):
            prefetcher.reportDataArrival(event)
        elif isinstance(event, PredictionResponseMatch):
            counter += 1
            # print(f"{counter} hardware prediction match: {event}\n")
            hardwarePredictionMatches.append(event)
        elif isinstance(event, BackwardsHit):
            #print(f"hardware backwards hit {event}\n")
            hardwareBackwardsHit.append(event)
        elif isinstance(event, BackwardsAddition):
            # print(f"\nhardware backwards addition {event}")
            hardwareBackwardsAddition.append(event)
        elif isinstance(event, TimelinessRdResp):
            # print(f"{counter} hardware TimelinessRdResp {event}\n")
            hardwareTimelinessResponse.append(event)
        elif isinstance(event, PredictionReplacement):
            hardwarePredictionReplacement.append(event)
            # print(f"hardware prediction replacement {event}\n")
        elif isinstance(event, TimelinessReplacement):
            #print(f"hardware timliness replacement {event}\n")
            hardwareTimelinessReplacement.append(event)
        elif isinstance(event, PredictionDecrease):
            # print(f"hardware prediction decrease {event}\n")
            hardwarePredictionDecrease.append(event)
        elif isinstance(event, PredictionMatchUpdate):
            print(f"hardware prediction match +1 {event}\n")
            hardwarePredictionMatchUpdate.append(event)
        # else:
        #     pass
        #     raise Exception()
    return prefetcher.reportStatistics()

In [210]:
softwarePredictionMatches = []
hardwarePredictionMatches = []

softwareBackwardsAddition = []
hardwareBackwardsAddition = []

softwareBackwardsHit = []
hardwareBackwardsHit = []

softwareTimelinessResponse = []
hardwareTimelinessResponse = []

hardwarePredictionReplacement = []
softwarePredictionReplacement = []

softwareTimelinessReplacement = []
hardwareTimelinessReplacement = []

softwarePredictionDecrease = []
hardwarePredictionDecrease = []

softwarePredictionMatchUpdate = []
hardwarePredictionMatchUpdate = []

prefetcher = BaseBackwardsPrefetcher(8, 1, 1, 2, 8, 4, True, 8, 8)
run_prefetcher(prefetcher)
print(f" stats: {prefetcher.reportStatistics()}")

WEIRDNESS addr[3:0] != 0, clock_time 44596.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 44598.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 44825.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 44854.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 47419.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 47892.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 47908.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 47947.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 47953.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 48001.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 48007.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 48055.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 48061.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 48109.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 48115.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 48163.0 request_addr 1
WEIRDNESS addr[3:0] != 0, clock_time 481

ZeroDivisionError: division by zero

In [195]:
def compareLists(hardware, software,attrs):
    match_counter = 0
    missmatch_counter = 0
    for i in range(len(hardware)):
        is_match = True
        for attr in attrs:
            if getattr(hardware[i], attr) != getattr(software[i], attr):
                is_match = False

        if not is_match:
            print(f"{i} hardware {hardware[i]}")
            print(f"{i} software {software[i]}\n")
            missmatch_counter += 1
        else:
            match_counter += 1
    print(f"match counter {match_counter}, missmatch {missmatch_counter}, hardware len {len(hardware)}, software len {len(software)}") 

In [196]:
compareLists(hardwareBackwardsAddition, softwareBackwardsAddition, ["parentVirtBase", "parentOffset", "idxTag"])

match counter 29274, missmatch 0, hardware len 29274, software len 29274


In [197]:
compareLists(hardwareBackwardsHit, softwareBackwardsHit, ["parentVirtBase", "parentOffset", "childOffset"])

match counter 3854, missmatch 0, hardware len 3854, software len 3854


In [198]:
compareLists(hardwareTimelinessReplacement, softwareTimelinessReplacement, ["idx", "tag", "pcHash", "repResp"])

match counter 76172, missmatch 0, hardware len 76172, software len 76172


In [199]:
compareLists(hardwareTimelinessResponse, softwareTimelinessResponse, ["valid", "tag", "pcHash", "replacementWay"])

match counter 3825, missmatch 0, hardware len 3825, software len 3825


In [200]:
compareLists(hardwarePredictionDecrease, softwarePredictionDecrease, ["idx", "tag", "oldConfidence", "oldParentOffset", "oldChildOffset"])

match counter 1857, missmatch 0, hardware len 1857, software len 1857


In [201]:
compareLists(hardwarePredictionReplacement, softwarePredictionReplacement, ["idx", "tag", "newParentOffset", "newChildOffset"])

match counter 1498, missmatch 0, hardware len 1498, software len 1498


In [202]:
compareLists(hardwarePredictionMatchUpdate, softwarePredictionMatchUpdate, ["idx", "tag", "oldConfidence"],)

match counter 470, missmatch 0, hardware len 470, software len 470


In [203]:
compareLists(hardwarePredictionMatches, softwarePredictionMatches, ["predIdxTag", "parentOffset", "childOffset", "confidence", "virtBase"],)

match counter 6728, missmatch 0, hardware len 6728, software len 6728


In [129]:
def time_diff(hardware, software):
    time_diffs = []
    for i in range(max(len(hardware), len(software))):
        time_diffs.append(int(hardware[i].clock_time)/10 - software[i].clock_time)
    return time_diffs

In [ ]:
timelinessWayResultsFalse = []
timelinessWayX= [1,2,4,8,16,32]
for i in timelinessWayX:
    prefetcher = BaseBackwardsPrefetcher(i, 1, 1, 3, 12, 4, False)
    timelinessWayResultsFalse.append(run_prefetcher(prefetcher))
    print(f"{i} stats: {prefetcher.reportStatistics()}")

In [ ]:
plt.bar(timelinessWayX, [r[0] for r in timelinessWayResultsFalse])
plt.xlabel('Number of timeliness ways')
plt.ylabel('Coverage')
plt.title("Coverage")

In [ ]:
plt.bar(timelinessWayX, [r[1] for r in timelinessWayResultsFalse])
plt.xlabel('Number of timeliness ways')
plt.ylabel('Accuracy')

In [ ]:
plt.bar(timelinessWayX, [r[-1] for r in timelinessWayResultsFalse])
plt.xlabel('Number of timeliness ways')
plt.ylabel('Median timeliness')

# Timeliness ways - report access prediction table

In [ ]:
timelinessWayResultsFalse = []
timelinessWayX= [1,2,4,8,16,32]
for i in timelinessWayX:
    prefetcher = BaseBackwardsPrefetcher(i, 1, 1, 3, 12, 4, True)
    timelinessWayResultsFalse.append(run_prefetcher(prefetcher))
    print(f"{i} stats: {prefetcher.reportStatistics()}")

In [ ]:
plt.bar(timelinessWayX, [r[0] for r in timelinessWayResultsFalse])
plt.xlabel('Number of timeliness ways')
plt.ylabel('Coverage')

In [ ]:
plt.bar(timelinessWayX, [r[1] for r in timelinessWayResultsFalse])
plt.xlabel('Number of timeliness ways')
plt.ylabel('Accuracy')

In [ ]:
plt.bar(timelinessWayX, [r[-1] for r in timelinessWayResultsFalse])
plt.xlabel('Number of timeliness ways')
plt.ylabel('Median timeliness')

## Confidence

In [ ]:
confidencethresholdresults = []
confidencethresholds= [1,2,3,4,5,6]
for i in confidencethresholds:
    prefetcher = BaseBackwardsPrefetcher(16, 1, 1, i, 12, 4, False)
    confidencethresholdresults.append(run_prefetcher(prefetcher))
    print(f"{i} stats: {prefetcher.reportStatistics()}")

In [ ]:
plt.bar(confidencethresholds, [r[0] for r in confidencethresholdresults])
plt.xlabel('Confidence threshold')
plt.ylabel('Coverage')

In [ ]:
plt.bar(confidencethresholds, [r[1] for r in confidencethresholdresults])
plt.xlabel('Confidence threshold')
plt.ylabel('Accuracy')

In [ ]:
plt.bar(confidencethresholds, [r[-1] for r in confidencethresholdresults])
plt.xlabel('Confidence threshold')
plt.ylabel('Timeliness')

# Confidence threshold report access

In [ ]:
confidencethresholdresultsTrue = []
confidencethresholds= [1,2,3,4,5,6]
for i in confidencethresholds:
    prefetcher = BaseBackwardsPrefetcher(16, 1, 1, i, 12, 4, True)
    confidencethresholdresultsTrue.append(run_prefetcher(prefetcher))
    print(f"{i} stats: {prefetcher.reportStatistics()}")

In [ ]:
plt.bar(confidencethresholds, [r[0] for r in confidencethresholdresultsTrue])
plt.xlabel('Confidence threshold')
plt.ylabel('Coverage')

In [ ]:
plt.bar(confidencethresholds, [r[1] for r in confidencethresholdresultsTrue])
plt.xlabel('Confidence threshold')
plt.ylabel('Accuracy')

In [ ]:
plt.bar(confidencethresholds, [r[-1] for r in confidencethresholdresults])
plt.xlabel('Confidence threshold')
plt.ylabel('Timeliness')

In [ ]:
confidencethresholdresultsTruePredWays = []
confidencethresholds= [2,3,4,5,6]
for i in confidencethresholds:
    prefetcher = BaseBackwardsPrefetcher(16, 2, 1, i, 12, 4, True)
    confidencethresholdresultsTruePredWays.append(run_prefetcher(prefetcher))
    print(f"{i} stats: {prefetcher.reportStatistics()}")

In [ ]:
prefetcher = BaseBackwardsPrefetcher(16, 1, 1, 3, 12, 4)
for event in total_order_events:
    if isinstance(event, ReportAccessLog):
        prefetcher.reportRequest(event)
    elif isinstance(event, ReportDataArrival):
        prefetcher.reportDataArrival(event)
    else:
        raise Exception()
print(f"PredConfidence {2} stats: {prefetcher.reportStatistics()}")

In [ ]:
prefetcher = BaseBackwardsPrefetcher(16, 1, 1, 3, 12, 4)
for event in total_order_events[:300000]:
    if isinstance(event, ReportAccessLog):
        prefetcher.reportRequest(event)
    elif isinstance(event, ReportDataArrival):
        prefetcher.reportDataArrival(event)
    else:
        raise Exception()
print(f"PredConfidence {2} stats: {prefetcher.reportStatistics()}")

In [ ]:
prefetcher = BaseBackwardsPrefetcher(16, 1, 1, 3, 12, 4)
for event in total_order_events[:300000]:
    if isinstance(event, ReportAccessLog):
        prefetcher.reportRequest(event)
    elif isinstance(event, ReportDataArrival):
        prefetcher.reportDataArrival(event)
    else:
        raise Exception()
print(f"PredConfidence {2} stats: {prefetcher.reportStatistics()}")

In [ ]:
len(prefetcher.confidenceUpdateTable.keys())

In [ ]:
prefetcher = BaseBackwardsPrefetcher(8, 1, 1, 2, 12, 4)
for event in total_order_events[:300000]:
    if isinstance(event, ReportAccessLog):
        prefetcher.reportRequest(event)
    elif isinstance(event, ReportDataArrival):
        prefetcher.reportDataArrival(event)
    else:
        raise Exception()
print(f"PredConfidence {2} stats: {prefetcher.reportStatistics()}")

In [ ]:
prefetcher = BaseBackwardsPrefetcher(16, 1, 1, 4, 12, 4, True)
run_prefetcher(prefetcher)
print(f"stats: {prefetcher.reportStatistics()}")

In [15]:
lengths = []
for k,v in prefetcher.predictionTable.items():
    if len(v) > 0:
        lengths.append(v[0].parentOffset)

In [ ]:
np.median(lengths)

In [ ]:
np.log2(130816.0)

In [ ]:
128*(20+20+7)/8

## Prefetcher counts

In [ ]:
predictionCount = 0
for k,v in prefetcher.predictionTable.items():
    predictionCount += len(v)
predictionCount

In [ ]:
len(prefetcher.backwardsTable.keys())

In [ ]:
timelinessTableCount = 0
for k,v in prefetcher.timelinessTable.items():
    timelinessTableCount += len(v)
timelinessTableCount